In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
pdfs = list(Path("./raw/").glob("*.pdf"))

In [6]:
import ollama
import fitz

In [5]:
OLLAMA_URL = "http://localhost:11434"
TEXT_EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2-vision"

In [20]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader

In [22]:
embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
)

In [23]:
text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
)

In [28]:
from tqdm.auto import tqdm

In [29]:
def semantic_chunking():
    embeddings = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
    )
    docs = []
    for doc in pdfs:
        loader = PyPDFLoader(doc)

        docs.extend(loader.load())
        
    semantic_chunks = []
    for doc1 in tqdm(docs, desc="Splitting documents"):
        semantic_chunks.extend(text_splitter.split_documents([doc1]))

    return semantic_chunks

semantic_chunking()







Splitting documents:   0%|          | 0/371 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [31]:
import concurrent.futures
from tqdm import tqdm
# Ensure your Langchain/Ollama imports are here too

def load_single_pdf(pdf_path):
    """Helper function to load a single PDF."""
    loader = PyPDFLoader(pdf_path)
    return loader.load()

def chunk_single_doc(doc, text_splitter):
    """Helper function to chunk a single document."""
    return text_splitter.split_documents([doc])

def semantic_chunking(pdfs):
    embeddings = OllamaEmbeddings(
        model="bge-m3",
        base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
        embeddings, 
        breakpoint_threshold_type="percentile" 
    )
    
    docs = []
    # 1. Parallelize PDF Loading
    # ThreadPool is great here because file I/O releases the GIL.
    print("Starting parallel PDF loading...")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Map applies the function to the iterable concurrently
        results = list(tqdm(executor.map(load_single_pdf, pdfs), total=len(pdfs), desc="Loading PDFs"))
        for res in results:
            docs.extend(res)
            
    semantic_chunks = []
    
    # 2. Parallelize Semantic Chunking
    # The chunker makes HTTP requests to Ollama, so threads work well.
    # CAUTION: Set max_workers carefully (see note below).
    MAX_WORKERS = 16
    
    print(f"Starting parallel chunking with {MAX_WORKERS} workers...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit tasks to the executor
        futures = [executor.submit(chunk_single_doc, doc, text_splitter) for doc in docs]
        
        # as_completed yields futures as they finish, keeping the progress bar accurate
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(docs), desc="Splitting documents"):
            semantic_chunks.extend(future.result())

    return semantic_chunks

semantic_chunks_parallel = semantic_chunking(pdfs)

Starting parallel PDF loading...


Loading PDFs: 100%|██████████| 19/19 [00:29<00:00,  1.56s/it]


Starting parallel chunking with 16 workers...


Splitting documents: 100%|██████████| 371/371 [31:01<00:00,  5.02s/it] 


In [33]:
import pickle

# Save the chunks
with open("semantic_chunks_parallel.pkl", "wb") as f:
    pickle.dump(semantic_chunks_parallel, f)

# To load them later:
# with open("semantic_chunks_parallel.pkl", "rb") as f:
#     loaded_chunks = pickle.load(f)
